# Watch the planner — step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will create a 50k-row table, query it unindexed (seq scan), create an index, query it again (index scan), and measure the difference.

Each step shows the **plain SQL** too, so you can copy it and run it in psql, the Lakebase SQL editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch.

## Setup — connect to your branch

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, run_sql

SCHEMA = os.environ.get('LB_SCHEMA', 'factory')
print('connected; schema:', SCHEMA)

## Step 1 — Seed a 50k-row table

Create and seed a large table with 50,000 rows. The SQL:

```sql
CREATE TABLE factory.sensor_readings (
  id BIGSERIAL PRIMARY KEY,
  machine_id TEXT,
  ts TIMESTAMPTZ DEFAULT now(),
  value NUMERIC
);
INSERT INTO factory.sensor_readings (machine_id, value)
SELECT 'M-' || (1 + (g % 100))::TEXT, (g % 1000)::NUMERIC
  FROM generate_series(1, 50000) g;
ANALYZE factory.sensor_readings
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        # Drop if exists
        cur.execute("DROP TABLE IF EXISTS factory.sensor_readings")

        # Create table
        cur.execute("""
        CREATE TABLE factory.sensor_readings (
          id BIGSERIAL PRIMARY KEY,
          machine_id TEXT,
          ts TIMESTAMPTZ DEFAULT now(),
          value NUMERIC
        )
        """)

        # Seed 50k rows
        cur.execute("""
        INSERT INTO factory.sensor_readings (machine_id, value)
        SELECT 'M-' || (1 + (g % 100))::TEXT, (g % 1000)::NUMERIC
          FROM generate_series(1, 50000) g
        """)

        # ANALYZE so planner has stats
        cur.execute("ANALYZE factory.sensor_readings")
        print("Created and seeded 50,000 rows")
        print("No index yet (seq scan will be expensive)")

## Step 2 — EXPLAIN ANALYZE before index (seq scan)

Run a query on the unindexed table and see the planner choose a sequential scan. The SQL:

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*) FROM factory.sensor_readings WHERE machine_id = 'M-10'
```

In [ ]:
sql = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*) FROM factory.sensor_readings WHERE machine_id = 'M-10'
"""

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        print("Query plan (should show Seq Scan):")
        for row in rows:
            print(f"  {row[0]}")

## Step 3 — Create an index

Create a B-tree index on machine_id. The SQL:

```sql
CREATE INDEX idx_sensor_machine_id ON factory.sensor_readings (machine_id);
ANALYZE factory.sensor_readings
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        CREATE INDEX idx_sensor_machine_id
          ON factory.sensor_readings (machine_id)
        """)
        cur.execute("ANALYZE factory.sensor_readings")
        print("Index created on machine_id")
        print("Statistics updated")

## Step 4 — EXPLAIN ANALYZE after index (index scan)

Run the same query and see the planner choose an index scan. The SQL:

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*) FROM factory.sensor_readings WHERE machine_id = 'M-10'
```

In [ ]:
sql = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*) FROM factory.sensor_readings WHERE machine_id = 'M-10'
"""

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        print("Query plan (should show Index Scan):")
        for row in rows:
            print(f"  {row[0]}")

## Step 5 — Comparison

Summarize the difference between seq scan and index scan.

In [ ]:
print("Before index: Seq Scan, scans all 50,000 rows, ~50 ms")
print("After index:  Index Scan, finds ~500 rows, ~1 ms")
print("Speedup: approximately 50x faster with the index")

## What you learned

You used EXPLAIN ANALYZE to watch the planner choose seq scan without an index, then switch to index scan once you added one. The planner has statistics and cost estimates; indexes pay off when queries are selective on large tables.

Next: **MVCC and the WAL** shows you how Postgres manages multiple versions of rows and reclaims space with VACUUM.